### EA (eV) のヒストグラム

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- データ読み込み ---
df = pd.read_csv("olivineDataset_withEA.csv")
df_valid = df.dropna(subset=['EA (eV)'])

# --- ヒストグラムの描画 ---
plt.figure(figsize=(8, 5))
plt.hist(df_valid['EA (eV)'], bins=20, color='steelblue', edgecolor='white')

plt.xlabel('EA (eV)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Histogram of Electron Affinity (EA)', fontsize=14)
plt.tight_layout()
plt.show()


### PLS 回帰分析（目的変数: EA (eV)、記述子: id 以外の数値列）

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# --- データ読み込み ---
df = pd.read_csv("olivineDataset_withEA.csv")

# --- 記述子列の選択（Index#, Composition, EA (eV), S/U を除く数値列）---
exclude_cols = ['Index#', 'Composition', 'EA (eV)', 'S/U']
descriptor_cols = [c for c in df.columns if c not in exclude_cols]

# --- EA(eV)・記述子の欠損行を除去 ---
df_valid = df.dropna(subset=['EA (eV)'] + descriptor_cols)

X = df_valid[descriptor_cols].values
y = df_valid['EA (eV)'].values

# --- 標準化 ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- 交差検証で最適成分数を探索 ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
n_components_range = range(1, 11)
rmse_cv = []

for n in n_components_range:
    pls = PLSRegression(n_components=n)
    y_cv = cross_val_predict(pls, X_scaled, y, cv=kf)
    rmse_cv.append(np.sqrt(mean_squared_error(y, y_cv)))

best_n = list(n_components_range)[np.argmin(rmse_cv)]
print(f"最適成分数: {best_n}  (CV-RMSE: {min(rmse_cv):.4f} eV)")

# --- 最適成分数でモデル構築 ---
pls_best = PLSRegression(n_components=best_n)
pls_best.fit(X_scaled, y)
y_pred = pls_best.predict(X_scaled).ravel()

rmse_train = np.sqrt(mean_squared_error(y, y_pred))
r2_train   = r2_score(y, y_pred)

# --- プロット ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# CV-RMSE vs 成分数
axes[0].plot(n_components_range, rmse_cv, 'o-', color='steelblue')
axes[0].axvline(best_n, color='red', linestyle='--', label=f'best n={best_n}')
axes[0].set_xlabel('Number of PLS Components')
axes[0].set_ylabel('CV-RMSE (eV)')
axes[0].set_title('Cross-Validation RMSE vs Components')
axes[0].legend()

# Parity plot
lims = [min(y.min(), y_pred.min()) - 0.5, max(y.max(), y_pred.max()) + 0.5]
axes[1].scatter(y, y_pred, color='steelblue', edgecolors='white', s=40)
axes[1].plot(lims, lims, 'k--', linewidth=1)
axes[1].set_xlim(lims)
axes[1].set_ylim(lims)
axes[1].set_xlabel('Actual EA (eV)')
axes[1].set_ylabel('Predicted EA (eV)')
axes[1].set_title(f'PLS Regression (n_components={best_n})')
axes[1].text(0.05, 0.95,
             f"Train RMSE = {rmse_train:.3f} eV\n$R^2$ = {r2_train:.3f}",
             transform=axes[1].transAxes, fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

plt.tight_layout()
plt.show()


### 全サンプルへの予測 EA 適用（Composition 別プロット）

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- 全サンプル読み込み ---
df_all = pd.read_csv("olivineDataset_withEA.csv")

exclude_cols = ['Index#', 'Composition', 'EA (eV)', 'S/U']
descriptor_cols = [c for c in df_all.columns if c not in exclude_cols]

# --- 記述子が揃っているサンプルのみ予測対象 ---
df_pred = df_all.dropna(subset=descriptor_cols).copy()
X_all_scaled = scaler.transform(df_pred[descriptor_cols].values)
df_pred['Predicted EA (eV)'] = pls_best.predict(X_all_scaled).ravel()

# --- 実測値あり・なしで色分け ---
has_actual = df_pred['EA (eV)'].notna()
comp = df_pred['Composition'].values
y_pred_all = df_pred['Predicted EA (eV)'].values
y_actual    = df_pred['EA (eV)'].values
x_idx = np.arange(len(comp))

# --- プロット ---
fig, ax = plt.subplots(figsize=(16, 5))

ax.bar(x_idx[~has_actual], y_pred_all[~has_actual],
       color='steelblue', label='Predicted (EA unknown)')
ax.bar(x_idx[has_actual],  y_pred_all[has_actual],
       color='orange', alpha=0.85, label='Predicted (EA known)')
ax.scatter(x_idx[has_actual], y_actual[has_actual],
           color='red', s=25, zorder=5, label='Actual EA')

ax.set_xticks(x_idx)
ax.set_xticklabels(comp, rotation=90, fontsize=7)
ax.set_xlabel('Composition (Index#: ' +
              ', '.join(df_pred['Index#'].astype(str).values[:3]) + '...)',
              fontsize=10)
ax.set_ylabel('EA (eV)', fontsize=12)
ax.set_title(f'PLS-Predicted EA (eV) for All Samples  '
             f'[n_components={best_n}, total={len(df_pred)}]', fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print(f"\n予測対象サンプル数 : {len(df_pred)}")
print(f"  実測値あり (橙+赤): {has_actual.sum()}")
print(f"  実測値なし (青)   : {(~has_actual).sum()}")
